# KoELECTRA ONNX → TFLite int8 변환

**사전 준비:**
1. 이 노트북과 함께 `model.onnx` (1.6MB) 를 Colab에 업로드
2. 런타임 → 런타임 유형 변경 → CPU 또는 T4 GPU (둘 다 가능)

**결과물:** `model_full_integer_quant.tflite` → 다운로드해서 Android 앱에 사용

In [ ]:
# 1. 패키지 설치
!pip install -q onnx2tf tensorflow

In [ ]:
# 2. model.onnx 업로드
from google.colab import files

print('model.onnx 파일을 선택하세요')
uploaded = files.upload()
print('업로드 완료:', list(uploaded.keys()))

In [ ]:
# 3. ONNX → TFLite int8 변환
!onnx2tf -i model.onnx -o tflite_output -oiqt
print('\n변환 완료!')

In [ ]:
# 4. 결과 파일 확인
import os

for f in os.listdir('tflite_output'):
    path = f'tflite_output/{f}'
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f'{f:60s}  {size_mb:.1f} MB')

In [ ]:
# 5. int8 모델 추론 검증
import numpy as np
import tensorflow as tf

TFLITE_PATH = 'tflite_output/model_full_integer_quant.tflite'

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('입력 텐서:')
for d in input_details:
    print(f'  {d["name"]:30s}  shape={d["shape"]}  dtype={d["dtype"]}')

print('\n출력 텐서:')
for d in output_details:
    print(f'  {d["name"]:30s}  shape={d["shape"]}  dtype={d["dtype"]}')

In [ ]:
# 6. 테스트 문장으로 추론
from transformers import AutoTokenizer

ID2LABEL = {0: '기관사칭', 1: '금전요구', 2: '개인정보', 3: '복합', 4: '정상'}
MAX_LENGTH = 128
TEST_SENTENCE = '저는 서울중앙지검 수사관입니다. 안전계좌로 즉시 송금해 주세요.'

tokenizer = AutoTokenizer.from_pretrained('monologg/koelectra-base-v3-discriminator')
enc = tokenizer(
    TEST_SENTENCE,
    return_tensors='np',
    truncation=True,
    max_length=MAX_LENGTH,
    padding='max_length',
)

for detail in input_details:
    name = detail['name'].split('/')[-1].split(':')[0]
    for key in enc:
        if key in name or name in key:
            interpreter.set_tensor(detail['index'], enc[key].astype(np.int32))
            break

interpreter.invoke()
logits = interpreter.get_tensor(output_details[0]['index'])
pred   = ID2LABEL[int(np.argmax(logits))]

print(f'입력: "{TEST_SENTENCE}"')
print(f'판정: {pred}  (예상: 복합)')

In [ ]:
# 7. 다운로드
from google.colab import files

files.download(TFLITE_PATH)
print('다운로드 완료 — Android 프로젝트의 assets/ 폴더에 넣으세요')